# Carga de datos

In [38]:
import pandas as pd
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from imblearn.pipeline import Pipeline as imbPipeline

#cargar los datos tratados en la parte 1
datos = pd.read_csv('https://raw.githubusercontent.com/owaruuu/desafio-telecom-x-oracle-latam-2/refs/heads/main/datos_tratados.csv')
datos.sample(5)

,ID_Cliente,Churn,Genero,Adulto_Mayor,Pareja,Dependientes,Antiguedad_Meses,Servicio_Telefono,Lineas_Multiples,Servicio_Internet,...,Respaldo_Online,Proteccion_Dispositivo,Soporte_Tecnico,TV_Streaming,Peliculas_Streaming,Contrato,Facturacion_Sin_Papel,Metodo_Pago,Cargo_Mensual,Cargo_Total
818,1195-OIYEJ,1,Male,0,0,0,13,1,No,Fiber optic,...,No,No,No,Yes,Yes,Month-to-month,1,Electronic check,91.10,1135.70
5210,7346-MEDWM,0,Female,0,0,0,59,1,Yes,Fiber optic,...,Yes,Yes,No,No,No,Month-to-month,1,Electronic check,83.25,4949.10
2852,4079-ULGFR,0,Male,0,0,0,16,1,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,0,Mailed check,20.00,275.70
4773,6728-VOIFY,0,Female,0,1,0,63,1,No,Fiber optic,...,Yes,No,No,Yes,Yes,One year,1,Electronic check,96.00,6109.75
1492,2202-CUYXZ,1,Male,0,0,0,1,1,No,Fiber optic,...,No,Yes,No,Yes,No,Month-to-month,1,Electronic check,84.85,84.85


In [39]:
#Exploracion de datos
print(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID_Cliente              7032 non-null   object 
 1   Churn                   7032 non-null   int64  
 2   Genero                  7032 non-null   object 
 3   Adulto_Mayor            7032 non-null   int64  
 4   Pareja                  7032 non-null   int64  
 5   Dependientes            7032 non-null   int64  
 6   Antiguedad_Meses        7032 non-null   int64  
 7   Servicio_Telefono       7032 non-null   int64  
 8   Lineas_Multiples        7032 non-null   object 
 9   Servicio_Internet       7032 non-null   object 
 10  Seguridad_Online        7032 non-null   object 
 11  Respaldo_Online         7032 non-null   object 
 12  Proteccion_Dispositivo  7032 non-null   object 
 13  Soporte_Tecnico         7032 non-null   object 
 14  TV_Streaming            7032 non-null   

In [40]:
datos[['Antiguedad_Meses','Cargo_Mensual','Cargo_Total']].sample(5)

,Antiguedad_Meses,Cargo_Mensual,Cargo_Total
189,11,36.05,402.60
6182,1,49.80,49.80
3441,61,61.45,3751.15
2586,45,20.40,930.45
4516,4,77.85,299.20


In [41]:
#Eliminacion de datos innecesarios
drop_datos = datos.drop(columns=['ID_Cliente'])

#Separacion de datos en Variables Explicativas y Variable Respuesta
X = drop_datos.drop(columns=['Churn'])
y = drop_datos['Churn']


In [42]:
print('Genero: ',drop_datos['Genero'].unique())
print('Adulto mayor: ',drop_datos['Adulto_Mayor'].unique())
print('Pareja: ',drop_datos['Pareja'].unique())
print('Dependientes: ',drop_datos['Dependientes'].unique())
print('Servicio telefono: ',drop_datos['Servicio_Telefono'].unique())
print('Lineas multiples: ',drop_datos['Lineas_Multiples'].unique())
print('Servicio internet: ',drop_datos['Servicio_Internet'].unique())
print('Seguridad online: ',drop_datos['Seguridad_Online'].unique())
print('Respaldo online: ',drop_datos['Respaldo_Online'].unique())
print('Proteccion dispositivo: ',drop_datos['Proteccion_Dispositivo'].unique())
print('Soporte tecnico: ',drop_datos['Soporte_Tecnico'].unique())
print('TV streaming: ',drop_datos['TV_Streaming'].unique())
print('Peliculas streaming: ',drop_datos['Peliculas_Streaming'].unique())
print('Contrato: ',drop_datos['Contrato'].unique())
print('Facturacion sin papel: ',drop_datos['Facturacion_Sin_Papel'].unique())
print('Metodo pago: ',drop_datos['Metodo_Pago'].unique())

Genero:  ['Female' 'Male']
Adulto mayor:  [0 1]
Pareja:  [1 0]
Dependientes:  [1 0]
Servicio telefono:  [1 0]
Lineas multiples:  ['No' 'Yes' 'No phone service']
Servicio internet:  ['DSL' 'Fiber optic' 'No']
Seguridad online:  ['No' 'Yes' 'No internet service']
Respaldo online:  ['Yes' 'No' 'No internet service']
Proteccion dispositivo:  ['No' 'Yes' 'No internet service']
Soporte tecnico:  ['Yes' 'No' 'No internet service']
TV streaming:  ['Yes' 'No' 'No internet service']
Peliculas streaming:  ['No' 'Yes' 'No internet service']
Contrato:  ['One year' 'Month-to-month' 'Two year']
Facturacion sin papel:  [1 0]
Metodo pago:  ['Mailed check' 'Electronic check' 'Credit card (automatic)'
 'Bank transfer (automatic)']


In [43]:
#Encoding de datos
columnas = X.columns

#Crear transformador de columnas
one_hot = make_column_transformer((OneHotEncoder(drop='if_binary'),['Genero','Lineas_Multiples','Servicio_Internet','Seguridad_Online','Respaldo_Online','Proteccion_Dispositivo','Soporte_Tecnico','TV_Streaming','Peliculas_Streaming','Contrato','Metodo_Pago']),
                                  remainder='passthrough',
                                  sparse_threshold=0,
                                  force_int_remainder_cols=False
                                  )

label_encoder = LabelEncoder()

X = one_hot.fit_transform(X)
one_hot.get_feature_names_out(columnas)

# pd.DataFrame(X,columns=one_hot.get_feature_names_out(columnas)).head(5)
y = label_encoder.fit_transform(y)


In [44]:
#Calcular la proporción de clientes que se dan de baja
#Evaluamos desbalanceo.
proporcion_clase = datos['Churn'].value_counts()
proporcion_clase_normalizada = datos['Churn'].value_counts(normalize=True)
print("Proporción de clases:\n", proporcion_clase)
print()
print("Proporción de clases normalizada:\n", proporcion_clase_normalizada)
print()
print("Tasa de evasion: ", proporcion_clase[1] / (proporcion_clase[0] + proporcion_clase[1]))

Proporción de clases:
 Churn
0    5163
1    1869
Name: count, dtype: int64

Proporción de clases normalizada:
 Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64

Tasa de evasion:  0.26578498293515357


In [45]:
#Probar diferentes metodos de balanceo
modelo = DecisionTreeClassifier(max_depth=10)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)
scoring = ['recall', 'precision', 'f1', 'roc_auc']

#Normal Model
resultados = cross_validate(modelo, X, y, cv=skf, scoring=scoring)
resultados_df = pd.DataFrame(resultados)
resultados_df.mean()

,0
fit_time,0.036679
score_time,0.010075
test_recall,0.550045
test_precision,0.552928
test_f1,0.551358
test_roc_auc,0.752506


In [46]:
#Oversample Model
pipeline = imbPipeline([
    ('smote', SMOTE(random_state=5)),
    ('model', DecisionTreeClassifier(max_depth=10, random_state=5))
])

oversample_resultados = cross_validate(
    pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring
)

resultados_df = pd.DataFrame(oversample_resultados)
resultados_df.mean()

,0
fit_time,0.105190
score_time,0.011920
test_recall,0.569275
test_precision,0.569073
test_f1,0.568410
test_roc_auc,0.788563


In [47]:
#Undersample Model
pipeline = imbPipeline([
    ('under', NearMiss(version=3)),
    ('model', DecisionTreeClassifier(max_depth=10, random_state=5))
])

undersample_resultados = cross_validate(pipeline, X, y, cv=skf, scoring=scoring)

resultados_df = pd.DataFrame(undersample_resultados)
resultados_df.mean()

,0
fit_time,0.138484
score_time,0.016151
test_recall,0.659740
test_precision,0.467379
test_f1,0.546878
test_roc_auc,0.722920


In [50]:
#Pruebo ocupando un modelo linear con valores escalados para comparar
numerical_cols = ['Antiguedad_Meses','Cargo_Mensual','Cargo_Total']

type(X)

# preprocessor = make_column_transformer(
#     (MinMaxScaler(), numerical_cols),
#     remainder='passthrough'
# )

# pipeline = Pipeline([
#     ('preprocess', preprocessor),
#     ('model', LogisticRegression())
# ])

# logistic_reg_resultados = cross_validate(pipeline, X, y, cv=skf, scoring=scoring)

# resultados_df = pd.DataFrame(undersample_resultados)
# resultados_df.mean()


numpy.ndarray

In [ ]:
def resumen_cv(nombre_modelo, cv_resultados):
    df = pd.DataFrame(cv_resultados).filter(like='test_')
    return pd.Series({
        'Model': nombre_modelo,
        'Recall': df['test_recall'].mean(),
        'Precision': df['test_precision'].mean(),
        'F1': df['test_f1'].mean(),
        'ROC_AUC': df['test_roc_auc'].mean()
    })

tabla_final = pd.DataFrame([
    resumen_cv("Original", resultados),
    resumen_cv("Oversample", oversample_resultados),
    resumen_cv("Undersample", undersample_resultados)
])

tabla_final

#Podemos apreciar una mejora en 'recall' para el modelo undersample pero a cambio de una peor 'precision'